# RFT Code Repair: MAI-Code-1.1-Flash (preview)

Use the **Microsoft Foundry Project SDK** to train `MAI-Code-1.1-Flash` with reinforcement fine-tuning (RFT). The model proposes Python code repairs, and a model-based grader rewards functional, minimal changes.

### What you'll build

| Stage | What you'll learn |
| :--- | :--- |
| Prepare | Understand the repair tasks and define a model-based reward. |
| Train | Upload data and create a managed RFT job with the Project SDK. |
| Inspect | Monitor the job and read its training metrics. |
| Deploy | Create a serving deployment with the Azure management SDK and wait for provisioning. |
| Infer | Send a new repair request to the fine-tuned deployment. |

MAI-Code-1.1-Flash supports **managed RFT only**, not SFT, DPO, or interactive training.

Follow the [setup guide](../../../GETTING_STARTED.md) and select the **Foundry managed** kernel. Run from this demo directory or its `notebooks/` directory. Sections 1-3 are local; Sections 4 onward use Foundry. Do not use **Run All** as an offline check.

> **Costs:** This workflow uses paid cloud resources and may incur charges. Training, model grading, deployment hosting, and inference have separate costs. You can stop after Section 8 for the training-only workflow.

## 1. Set the model and paths

Use `MAI-Code-1.1-Flash` as the training model ID. Confirm preview access, model and grader availability, region support, and quota in your project before running the remote sections.

In [ ]:
import json
import os
import time
from pathlib import Path

MODEL = "MAI-Code-1.1-Flash"

DEMO_ROOT = Path.cwd().resolve()
if DEMO_ROOT.name == "notebooks":
    DEMO_ROOT = DEMO_ROOT.parent

TRAIN_PATH = DEMO_ROOT / "data" / "preserved" / "code-repair-rft-training.jsonl"
VALIDATION_PATH = DEMO_ROOT / "data" / "preserved" / "code-repair-rft-validation.jsonl"
OUTPUT_DIR = DEMO_ROOT / "outputs"

## 2. Inspect a code-repair task

The [dataset](../data/README.md) has **50 training** and **15 validation** tasks. Each record contains:

| Field | Purpose |
| :--- | :--- |
| `messages` | Developer instructions and the user's code-repair request, sent to the model. |
| `reference_patch` | A reference diff for the grader, not the model. |
| `test_expectations` | Expected behavior for the grader to consider. |
| `requirements` | Repair constraints and edge cases for the grader. |

There is no assistant answer in the data: RFT generates candidate repairs and scores them. The prompt asks for JSON containing `patch`, `explanation`, and `tests`. Let's inspect the first example.

In [ ]:
training_rows = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines()]
validation_rows = [json.loads(line) for line in VALIDATION_PATH.read_text(encoding="utf-8").splitlines()]

print(f"Training examples: {len(training_rows)}; validation examples: {len(validation_rows)}")
print(json.dumps(training_rows[0], indent=2))

## 3. Load the score-model grader

RFT needs a **reward function**, not a stored assistant answer. This sample uses a `score_model` grader: a second model, `o3-mini-2025-01-31`, reviews repairs generated by the MAI model.

### How a response becomes a reward

1. **Generate:** the training model receives only the record's `messages` and proposes a repair.
2. **Build the grading prompt:** Foundry substitutes the reference fields and candidate response into the grader's message templates.
3. **Score:** the grading model follows the rubric and returns one floating-point score from **0 to 1**.
4. **Learn:** the training service uses rewards to update the coding model toward higher-scoring responses. The grading model is not trained by this job.

### What the grader rewards

| Criterion | Weight | What the reviewer looks for |
| :--- | :--- | :--- |
| Functional correctness | 55% | The patch satisfies the stated test expectations. |
| Requirements and edge cases | 20% | The repair covers the constraints and boundary cases. |
| Minimality and regression safety | 15% | Changes are focused, maintainable, and avoid unrelated behavior changes. |
| Patch and explanation | 10% | A usable unified diff and a concise explanation supported by the patch. |

These weights are instructions in **one grading prompt**, not four separately executed tests. The [JSON definition](../graders/code_repair_quality.json) contains the full rubric.

| Grader setting | Meaning |
| :--- | :--- |
| `model` | `o3-mini-2025-01-31`, the reviewer rather than the model being trained. |
| `input` | System and user messages containing the rubric, references, and candidate. |
| `{{item.reference_patch}}` | A reference diff from the current dataset record. |
| `{{item.test_expectations}}` / `{{item.requirements}}` | Expected behavior and repair constraints from that record. |
| `{{sample.output_text}}` | The complete candidate response, including its JSON-as-text patch and explanation. |
| `range` / `pass_threshold` | Score bounds `[0, 1]` and a passing threshold of `0.7`; neither is an executable-test pass rate. |
| `sampling_params.seed` | `42` for grading-model sampling, not a training seed or a guarantee of identical scores. |

> **Reward quality matters:** the grader does not apply patches or execute tests. Before relying on its scores, calibrate it with known-correct repairs, incorrect or empty patches, malformed responses, and attempts to manipulate the reviewer. Model grading also adds work and cost for sampled responses.

In [ ]:
GRADER = json.loads(
    (DEMO_ROOT / "graders" / "code_repair_quality.json").read_text(encoding="utf-8")
)
GRADER

## 4. Connect to your Foundry project

Copy [`.env.template`](../.env.template) to a demo-local `.env` and set `FOUNDRY_PROJECT_ENDPOINT`. Authenticate with a provider supported by `DefaultAzureCredential`, such as Azure CLI.

`AIProjectClient.get_openai_client()` provides the project's Files and Fine-tuning APIs. We disable automatic retries so a lost upload or submission response is not silently retried. Check Foundry before repeating either operation after an error.

This sample uses **Global training**. MAI supports Global and Developer, not Standard or Data zone; these training types do not guarantee residency in the project's region. Review the [model and training-type support](https://learn.microsoft.com/azure/foundry/fine-tuning/overview#supported-models) and data terms before uploading.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(DEMO_ROOT / ".env", override=False)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
client = project_client.get_openai_client(max_retries=0)
OUTPUT_DIR.mkdir(exist_ok=True)

## 5. Upload the training and validation files

Upload both JSONL files with `purpose="fine-tune"`, then wait for processing. The files include the grader's reference fields; do not strip them before upload.

Each execution uploads new files. Their IDs are saved immediately in ignored `outputs/` files. If interrupted, retrieve those files with `client.files.retrieve(...)` instead of uploading again. If file quota is full, reconcile existing files rather than deleting shared data.

In [ ]:
(OUTPUT_DIR / "project-endpoint.txt").write_text(
    PROJECT_ENDPOINT, encoding="utf-8"
)
with TRAIN_PATH.open("rb") as data:
    training_file = client.files.create(file=data, purpose="fine-tune")
(OUTPUT_DIR / "training-file-id.txt").write_text(training_file.id, encoding="utf-8")

with VALIDATION_PATH.open("rb") as data:
    validation_file = client.files.create(file=data, purpose="fine-tune")
(OUTPUT_DIR / "validation-file-id.txt").write_text(validation_file.id, encoding="utf-8")

for uploaded_file in (training_file, validation_file):
    client.files.wait_for_processing(uploaded_file.id, max_wait_seconds=600)

print("Training and validation files are processed.")

## 6. Create the RFT job

Pass the uploaded file IDs and grader directly to `fine_tuning.jobs.create`. `method.type="reinforcement"` selects RFT. `trainingType="globalStandard"` selects Global training; omitting it defaults to Standard, which this model does not support.

### Available RFT controls

The RFT API exposes the controls below under `method.reinforcement.hyperparameters`. **Supported settings, accepted values, and defaults depend on the training model.** This sample omits overrides and uses the service's MAI defaults; the table is not a promise that every model accepts every control.

| Hyperparameter | Controls | Trade-off |
| :--- | :--- | :--- |
| `n_epochs` | Passes through the training dataset. | More passes add training work and can overfit a small dataset. |
| `batch_size` | Examples in a training batch. | Changes update frequency and variability. |
| `learning_rate_multiplier` | Scale of the learning rate. | Larger updates can learn faster but can also destabilize training. |
| `compute_multiplier` | Compute spent exploring candidate responses. | More exploration adds training work. |
| `eval_interval` | Training steps between validation runs. | More frequent validation gives earlier feedback but adds evaluation work. |
| `eval_samples` | Generated samples used for evaluation. | More samples add sampling and grading work. |
| `reasoning_effort` | Reasoning effort, where the selected model supports it. | Can change response length, latency, and cost. |

For a later experiment, a supported override belongs beside `grader`, for example:

```python
method={
    "type": "reinforcement",
    "reinforcement": {
        "grader": GRADER,
        "hyperparameters": {"learning_rate_multiplier": 1.0},
    },
}
```

The example illustrates placement, not a recommended MAI setting. Change one supported control at a time and compare validation reward and repair quality. Grader sampling settings live in `GRADER["sampling_params"]`, not in the training hyperparameters. See the [RFT configuration guide](https://learn.microsoft.com/azure/foundry/openai/how-to/reinforcement-fine-tuning#configure-hyperparameters) and review training and grading [pricing](https://aka.ms/oai/pricing) before submission.

**Run this cell once to create a new paid job.** To continue an existing job, skip this cell and use its saved ID in Section 7. If submission raises an error, check Foundry for the job before trying again.

In [ ]:
training_job = client.fine_tuning.jobs.create(
    model=MODEL,
    training_file=training_file.id,
    validation_file=validation_file.id,
    method={"type": "reinforcement", "reinforcement": {"grader": GRADER}},
    extra_body={"trainingType": "globalStandard"},
    suffix="mai-code-repair",
)
(OUTPUT_DIR / "job-id.txt").write_text(training_job.id, encoding="utf-8")
print(f"Job status: {training_job.status}")

## 7. Monitor training

Retrieve the job every minute until it succeeds, fails, or is cancelled. The six-hour timeout stops local waiting, **not** the remote job or its costs.

After a kernel restart, run Sections 1 and 4 with the **same project**, then continue here using `outputs/job-id.txt`. Do not rerun upload or submission to restart monitoring. For a job recovered through Foundry, save its confirmed ID to that file first.

In [ ]:
job_id = (OUTPUT_DIR / "job-id.txt").read_text(encoding="utf-8").strip()
deadline = time.monotonic() + 6 * 60 * 60

while True:
    job = client.fine_tuning.jobs.retrieve(job_id)
    print(f"Job status: {job.status}")
    if job.status in {"succeeded", "failed", "cancelled"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Monitoring timed out. The remote job may still be running.")
    time.sleep(60)

if job.status != "succeeded":
    raise RuntimeError(f"Training {job.status}: {job.error}")

(OUTPUT_DIR / "completed-job.json").write_text(job.model_dump_json(indent=2), encoding="utf-8")
print("Training completed. The model and result-file IDs are saved in outputs/completed-job.json.")

## 8. Inspect training metrics

`job.result_files` contains the service-produced training metrics. Download the CSVs with the Files API and view them as tables.

| Signal | How to interpret it |
| :--- | :--- |
| Training mean reward | How sampled training responses scored against the grader. Batches differ, so look at the trend. |
| Validation mean reward | Scores on the 15 validation tasks used during optimization, not an untouched test set. |
| Diverging training and validation rewards | A reason to inspect overfitting, grader behavior, and individual repairs. |

Column names and metric availability depend on the service response. Rewards are not executable-test pass rates or proof of improvement on unseen code.

In [ ]:
import pandas as pd
from IPython.display import display

if not job.result_files:
    raise RuntimeError("The service returned no training metrics files.")

for index, file_id in enumerate(job.result_files, start=1):
    result = client.files.content(file_id)
    metrics_path = OUTPUT_DIR / f"training-metrics-{index}.csv"
    result.write_to_file(metrics_path)
    display(pd.read_csv(metrics_path))

## 9. Deploy the fine-tuned model

Training creates a **model artifact**, not an inference endpoint. This optional section creates the serving deployment directly with `CognitiveServicesManagementClient`, using the same `DefaultAzureCredential` as training. The management SDK is included in the central environment; the Project SDK client is used for inference in Section 10.

| Identifier | Used for |
| :--- | :--- |
| `MODEL` | The base-model version submitted for training. |
| `job.fine_tuned_model` | The resulting artifact to deploy, also recorded in `outputs/completed-job.json`. |
| Deployment name | The serving endpoint's name, passed as `model` on an inference request. |

### Configure the serving target

Set these deployment-only values in your local [`.env`](../.env.template):

| Setting | Value |
| :--- | :--- |
| `FOUNDRY_SUBSCRIPTION_ID` | Subscription containing the existing Foundry resource. |
| `FOUNDRY_RESOURCE_GROUP` | Resource group containing that resource. |
| `FOUNDRY_ACCOUNT_NAME` | Parent resource of the project used for training, not the project name. |
| `FOUNDRY_CODE_REPAIR_DEPLOYMENT` | A new deployment name, not an existing shared deployment. |
| `FOUNDRY_CODE_REPAIR_CAPACITY` | A positive capacity allocation within your model's available quota. Capacity units are model-specific. |

The cell checks that the configured resource contains the training project before creating a deployment. Your identity needs project/deployment read access and `Microsoft.CognitiveServices/accounts/deployments/write`, such as the **Foundry Owner** role, plus available serving quota. It does not grant permissions or provision a new Foundry resource.

The request uses `job.fine_tuned_model`, model format `Microsoft`, fine-tuned artifact version `1`, and SKU `GlobalStandard`. Fine-tuned MAI-Code-1.1-Flash supports **Serverless Global Standard**, not Managed Compute or Fireworks; see [model compatibility](https://learn.microsoft.com/azure/foundry/fine-tuning/overview#supported-models). Training type and serving SKU are separate settings.

`begin_create_or_update(...)` starts the deployment and `.result()` waits for provisioning to finish. This sample refuses to update an existing deployment. The requested name is saved before creation; if interrupted or the response is lost, open a new management client and inspect that name with `deployments.get(...)` before retrying. A successful response is saved to `outputs/completed-deployment.json`.

> **Serving costs:** running this cell creates a paid serving resource. Review [hosting/inference pricing](https://aka.ms/oai/pricing) and quota first. Hosting can continue while the notebook is idle; closing Python clients does not remove the deployment.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import (
    Deployment, DeploymentModel, DeploymentProperties, Sku,
)

load_dotenv(DEMO_ROOT / ".env", override=True)
subscription_id = os.environ["FOUNDRY_SUBSCRIPTION_ID"]
resource_group = os.environ["FOUNDRY_RESOURCE_GROUP"]
account_name = os.environ["FOUNDRY_ACCOUNT_NAME"]
DEPLOYMENT_NAME = os.environ["FOUNDRY_CODE_REPAIR_DEPLOYMENT"]
capacity = int(os.environ["FOUNDRY_CODE_REPAIR_CAPACITY"])
if capacity <= 0:
    raise ValueError("Choose a positive deployment capacity within your available quota.")
if job.status != "succeeded" or not job.fine_tuned_model:
    raise RuntimeError("Training must succeed and produce a model artifact before deployment.")

with CognitiveServicesManagementClient(credential, subscription_id) as management_client:
    project_endpoint = PROJECT_ENDPOINT.rstrip("/")
    target_project = management_client.projects.get(
        resource_group, account_name, project_endpoint.rsplit("/", 1)[-1]
    )
    if project_endpoint not in {
        endpoint.rstrip("/") for endpoint in (target_project.properties.endpoints or {}).values()
    }:
        raise ValueError("Choose the parent resource of FOUNDRY_PROJECT_ENDPOINT.")
    existing_deployments = management_client.deployments.list(resource_group, account_name)
    if any(item.name.casefold() == DEPLOYMENT_NAME.casefold() for item in existing_deployments):
        raise ValueError("Choose a new deployment name; this sample does not update existing deployments.")

    (OUTPUT_DIR / "deployment-name.txt").write_text(DEPLOYMENT_NAME, encoding="utf-8")
    deployment = management_client.deployments.begin_create_or_update(
        resource_group, account_name, DEPLOYMENT_NAME,
        Deployment(
            sku=Sku(name="GlobalStandard", capacity=capacity),
            properties=DeploymentProperties(
                model=DeploymentModel(format="Microsoft", name=job.fine_tuned_model, version="1")
            ),
        ),
    ).result()
    if (
        deployment.properties.provisioning_state != "Succeeded"
        or deployment.properties.model.name != job.fine_tuned_model
    ):
        raise RuntimeError("Deployment is not ready with this job's fine-tuned model.")

(OUTPUT_DIR / "completed-deployment.json").write_text(
    json.dumps(deployment.as_dict(), indent=2), encoding="utf-8"
)
print("Fine-tuned deployment provisioning completed.")

## 10. Run inference on a new repair request

Use the project's OpenAI-compatible client to call the **deployment name**, not `MODEL` or the fine-tuned artifact ID. The request below uses the same response contract as the training tasks: JSON containing `patch`, `explanation`, and `tests`.

- Send the repair instructions and buggy code as `messages`.
- Do **not** send a reference patch or grader-only fields to the coding model.
- `max_completion_tokens` limits generated tokens for this request; it is separate from training hyperparameters and grader sampling settings.
- Review the response as text. This cell does not apply the patch or run generated tests.

This is a single-prompt inference demonstration, not a held-out evaluation or a base-versus-fine-tuned comparison.

In [ ]:
inference_messages = [
    {
        "role": "developer",
        "content": "Repair the Python code with minimal changes. Return only JSON with "
                   "patch (a unified diff), explanation, and tests fields.",
    },
    {
        "role": "user",
        "content": "Fix take_prefix so count=0 returns an empty list, negative counts raise "
                   "ValueError, and the input list is not modified.\n\n"
                   "```python\ndef take_prefix(items, count):\n"
                   "    return items[:count] if count else items\n```",
    },
]

response = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=inference_messages,
    max_completion_tokens=2048,
)
(OUTPUT_DIR / "inference-response.json").write_text(response.model_dump_json(indent=2), encoding="utf-8")
print(response.choices[0].message.content)

## 11. Close local clients and plan evaluation

The complete flow covers training, deployment, and one inference request. Keep saved IDs, job and inference responses, and metric CSVs in ignored `outputs/`, not in committed notebook output.

The 15 validation tasks are used during training, not held out for a final quality claim. Before relying on the model, calibrate the grader and compare repairs on an independent test set using isolated executable tests and human review.

### Resource handoff

- **Local clients:** close them below to release local connections. This does not stop remote resources.
- **Serving deployment:** if you created one in Section 9, review its costs and remove that specific deployment in Foundry when no longer needed. This is a separate, confirmed action; the notebook does not delete it.
- **Training inputs and artifacts:** retain the references needed for recovery and evaluation. Do not delete files shared by other jobs or a grader deployment owned by another workflow.

Follow [post-training guidance](../../../POST_TRAINING.md) for evaluation, serving, cancellation, and cleanup. If you stopped after training, skip Sections 9-10 and run only the client-close cell below.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.

In [ ]:
client.close()
project_client.close()
credential.close()